In [3]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END 
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages

In [ ]:
load_dotenv(override=True)

In [72]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [12]:
from youtube_transcript_api import YouTubeTranscriptApi, FetchedTranscript
def get_transcript(yt_link)-> FetchedTranscript : 
    ytt_api = YouTubeTranscriptApi()
    video_id = yt_link.split("v=")[1].split("&")[0]
    
    res = ytt_api.fetch(video_id)
    return res 


snippet = get_transcript("https://www.youtube.com/watch?v=F6nsDqXA1CY").snippets
    

In [15]:
def chunk_transcript(snippets, max_chunk_duration=45):
    chunks = []
    current_text = []
    chunk_start = snippets[0].start

    for snip in snippets:
        current_text.append(snip.text)
        if (snip.start + snip.duration) - chunk_start >= max_chunk_duration:
            chunks.append({
                "text": " ".join(current_text),
                "start": chunk_start
            })
            current_text = []
            chunk_start = snip.start + snip.duration

    if current_text:
        chunks.append({"text": " ".join(current_text), "start": chunk_start})

    return chunks

chunks = chunk_transcript(snippet , 60)


In [18]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
video_id = "F6nsDqXA1CY"
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
docs = [
    Document(page_content=c["text"], metadata={"start": c["start"], "video_id": video_id})
    for c in chunks
]
vectorstore = FAISS.from_documents(docs, embedding_model)

In [19]:
retriver = vectorstore.as_retriever(search_kwargs={"k" : 3})

In [21]:
retriver.invoke("who is teaching what")

[Document(id='64e8af55-b06b-4245-a4e2-784b16ad6a8b', metadata={'start': 0.56, 'video_id': 'F6nsDqXA1CY'}, page_content="Englishleap podcast >> from Speak English with Class. [music] >> Hey, English learners. Welcome back to the Englishleap podcast, your cozy little place to learn easy English through real everyday conversations. I'm Anna. >> And I'm Jake. >> How are you today, Jake? >> I'm okay, but I feel like today's topic is judging me. >> We haven't even said the topic yet. >> My body knows. My shoes know. Even my sofa knows. >> Your sofa knows? >> Yes. My sofa and I have been very close recently. >> Ah, that explains it. Because today we're talking about exercise. >> See, I knew it. >> But don't worry. This is not an episode about becoming a fitness expert overnight. >> Good, because I was not emotionally ready for that. >> This is about real exercise, normal exercise, walking, stretching, moving more, starting again, feeling tired,"),
 Document(id='4e870a8d-5447-4fe3-a973-e264c94

In [48]:
vector_store_global = {}

In [ ]:
# def load_yt_transcript_and_make_retiriver(state) : 
#     link = state["yt_link"]
#     snippet = get_transcript(link).snippets
#     chunks = chunk_transcript(snippet , 30)
#     embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
#     docs = [
#     Document(page_content=c["text"], metadata={"start": c["start"], "video_id": video_id})
#     for c in chunks
#     ]
#     vectorstore = FAISS.from_documents(docs, embedding_model)
#     vector_store_global[link] = vectorstore
#     return {
#         "is_first" : False
#     }
    

In [83]:
def ingest_video(yt_link) -> str:
    video_id = yt_link.split("v=")[1].split("&")[0]

    if video_id in vector_store_global:
        print("ingestion not done")
        return video_id

    snippets = get_transcript(yt_link).snippets
    chunks = chunk_transcript(snippets, 30)
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    docs = [
        Document(page_content=c["text"], metadata={"start": c["start"], "video_id": video_id})
        for c in chunks
    ]
    vector_store_global[video_id] = FAISS.from_documents(docs, embedding_model)
    return video_id

In [58]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [76]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage
def chat(state, config):
    video_id = config["configurable"]["video_id"]
    query = state["messages"][-1].content
    store = vector_store_global[video_id]
    retriever = store.as_retriever(search_kwargs={"k": 3})
    context = retriever.invoke(query)

    prompt = PromptTemplate(
        template="User Query: {query}\n\nContext: {context}\n\nAnswer based only on the context",
        input_variables=["query", "context"]
    )
    chain = prompt | llm | StrOutputParser()
    output = chain.invoke({"query": query, "context": context})

    return {"messages": [AIMessage(content=output)]}
    

In [ ]:
# def should_get_trans(state) : 
#     if state["is_first"] : 
#         return True 
#     return False

In [77]:
graph = StateGraph(ChatState)
graph.add_node("chat", chat)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)
checkpointer = InMemorySaver()
graph = graph.compile(checkpointer=checkpointer)

In [84]:
video_id = ingest_video("https://www.youtube.com/watch?v=F6nsDqXA1CY")

res1 = graph.invoke(
    {"messages": [HumanMessage(content="what level did you say it's for?")]},
    config={"configurable": {"thread_id": "test1", "video_id": video_id}}
)

ingestion not done


In [85]:
res1["messages"][-1].content

'This episode is perfect for B1 and B2 English learners. If you are A2, you can still watch it as a challenge.'

In [71]:
res1["messages"]

[HumanMessage(content='what is this video about ? and whats its goal ? ', additional_kwargs={}, response_metadata={}, id='50fcdc70-baae-4e9f-8a72-b03aa8a62459'),
 AIMessage(content='The video appears to be a language learning episode aimed at helping English learners, specifically those at the B1 and B2 levels, with options for A2 learners to join as a challenge. The goal of the video is to provide engaging content that encourages viewers to practice their English skills by watching the episode and then continuing their learning through the English Leap Club, where they can access additional resources like transcripts, quizzes, and games. The video emphasizes the importance of ongoing practice and suggests fun ways to incorporate learning into everyday life, stressing that viewers should enjoy the process rather than feeling overwhelmed.', additional_kwargs={}, response_metadata={}, id='8a2b3c8f-6431-4e34-bdf0-88ac8e9bae11', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content=